# CCTV Commercial II-01: full
Private CCTV processing. No known-speed reference or prior Kia/Mercedes correction is used.

In [ ]:
from pathlib import Path
import hashlib,json,subprocess,sys
sources={'cctv_inference.py': '"""Sequential, speed-blind measurements for long static CCTV recordings.\n\nEvery source frame is decoded; detection and cuboids are sampled independently.\nOnly cars with recent image movement contribute cuboids. No reference speeds,\nMercedes/Kia learned scales, or expected-speed clipping are used.\n"""\nimport argparse\nfrom collections import defaultdict, deque\nimport gzip\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\nimport urllib.request\n\nimport cv2\nimport numpy as np\nfrom scipy.optimize import linear_sum_assignment\nfrom ultralytics import YOLO\n\nfrom validation_inference import MODEL_REPO, MODEL_REVISION, MODEL_FILE, MODEL_SHA, overlaps\n\n\ndef region_for(point, profile, width, height):\n    for name, polygon in profile[\'regions\'].items():\n        contour = np.asarray(polygon, np.float32) * [width, height]\n        if cv2.pointPolygonTest(contour.astype(np.float32), tuple(map(float, point)), False) >= 0:\n            return name\n    return None\n\n\ndef inference(video, output, camera, duration=None, resume_s=0, device=\'0\'):\n    config = json.loads((Path(__file__).parent/\'cctv_camera_profiles.json\').read_text())\n    profile = config[\'cameras\'][camera]\n    geometry_hz=profile.get(\'geometry_hz\',config[\'geometry_hz\'])\n    motion_minimum_span=profile.get(\'motion_history_minimum_span_s\',.4)\n    video = Path(video); output = Path(output); output.mkdir(parents=True, exist_ok=True)\n    with video.open(\'rb\') as stream:\n        source_sha = hashlib.file_digest(stream, \'sha256\').hexdigest()\n    assert source_sha == profile[\'sha256\'], \'Unexpected source recording\'\n    cap = cv2.VideoCapture(str(video)); assert cap.isOpened()\n    width, height = [int(cap.get(key)) for key in [cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT]]\n    fps = float(cap.get(cv2.CAP_PROP_FPS)); total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n    stop_frame = total if duration is None else min(total, int(round(duration*fps)))\n    resume_frame = int(round(resume_s*fps))\n    models = output.parent/\'models\'; models.mkdir(exist_ok=True)\n    weights = models/\'urban_geometry.pt\'\n    if not weights.exists():\n        urllib.request.urlretrieve(f\'https://huggingface.co/{MODEL_REPO}/resolve/{MODEL_REVISION}/{MODEL_FILE}\', weights)\n    with weights.open(\'rb\') as stream:\n        assert hashlib.file_digest(stream, \'sha256\').hexdigest() == MODEL_SHA\n    tracker = YOLO(str(models/\'yolo11s.pt\'))\n    geometry = YOLO(str(weights))\n    tracker_config = output/\'bytetrack_10fps.yaml\'\n    tracker_config.write_text(\'tracker_type: bytetrack\\ntrack_high_thresh: 0.25\\ntrack_low_thresh: 0.10\\nnew_track_thresh: 0.25\\ntrack_buffer: 10\\nmatch_thresh: 0.80\\nfuse_score: True\\n\')\n    track_file = gzip.open(output/\'tracks.jsonl.gz\', \'wt\', encoding=\'utf-8\')\n    geom_file = gzip.open(output/\'observations.jsonl.gz\', \'wt\', encoding=\'utf-8\')\n    histories = defaultdict(deque)\n    counts = defaultdict(int)\n    next_detection, next_geometry = resume_frame, resume_s\n    start = time.monotonic(); index = 0; previous_pts = -1; timestamp_fallbacks = 0\n    checkpoints = [];ground_votes=defaultdict(list)\n    try:\n        while index < stop_frame:\n            ok = cap.grab()\n            if not ok:\n                break\n            pts = float(cap.get(cv2.CAP_PROP_POS_MSEC))/1000\n            expected = index/fps\n            timestamp = pts if np.isfinite(pts) and pts > previous_pts and abs(pts-expected) < .15 else expected\n            timestamp_fallbacks += int(timestamp != pts)\n            previous_pts = timestamp\n            if index < resume_frame or index + 1e-6 < next_detection:\n                index += 1; continue\n            ok, frame = cap.retrieve(); assert ok, f\'Cannot retrieve frame {index}\'\n            while next_detection <= index + 1e-6:\n                next_detection += fps/config[\'detection_hz\']\n            if counts[\'detection_frames\'] == 0:\n                cv2.imwrite(str(output/\'reference.jpg\'), frame)\n            detected = tracker.track(frame, persist=True, tracker=str(tracker_config), device=device,\n                                     imgsz=1280, classes=[2,3,5,7], conf=.10, half=True, verbose=False)[0]\n            boxes = detected.boxes.xyxy.cpu().numpy()\n            ids = detected.boxes.id.cpu().numpy().astype(int) if detected.boxes.id is not None else np.full(len(boxes), -1)\n            scores = detected.boxes.conf.cpu().numpy(); classes = detected.boxes.cls.cpu().numpy().astype(int)\n            # A reset at the resume boundary cannot silently join unrelated track IDs.\n            ids = np.where(ids >= 0, ids + (1000000 if resume_frame else 0), -1)\n            eligible = []\n            for k, (box, tid, score, cls) in enumerate(zip(boxes, ids, scores, classes)):\n                if tid < 0:\n                    continue\n                center = (box[:2] + box[2:])/2\n                region = region_for((center[0], box[3]), profile, width, height)\n                history = histories[int(tid)]\n                history.append((timestamp, center))\n                while history and timestamp-history[0][0] > 2.5:\n                    history.popleft()\n                displacement = float(np.linalg.norm(center-history[0][1]))\n                moving = timestamp-history[0][0] >= motion_minimum_span-.001 and displacement >= max(8, .10*(box[3]-box[1]))\n                row = dict(frame_index=index, time_s=timestamp, track_id=int(tid), class_id=int(cls),\n                           box=box.tolist(), confidence=float(score), region=region, moving=bool(moving))\n                track_file.write(json.dumps(row, separators=(\',\', \':\'))+\'\\n\'); counts[\'track_rows\'] += 1\n                if cls == 2 and moving and region and box[3]-box[1] >= 40:\n                    eligible.append(k)\n            counts[\'detection_frames\'] += 1\n            if timestamp >= next_geometry:\n                next_geometry = timestamp+1/geometry_hz-.01\n                if eligible:\n                    result = geometry.predict(frame, device=device, imgsz=1920, classes=[0], conf=.15,\n                                              half=True, verbose=False)[0]\n                    counts[\'geometry_frames\'] += 1\n                    if result.keypoints is not None and len(result.boxes):\n                        gb = result.boxes.xyxy.cpu().numpy(); gs = result.boxes.conf.cpu().numpy()\n                        kp = result.keypoints.data.cpu().numpy()\n                        overlap = overlaps(boxes[eligible], gb); ii, jj = linear_sum_assignment(1-overlap)\n                        for i,j in zip(ii,jj):\n                            if overlap[i,j] < .35:\n                                continue\n                            k = eligible[i]\n                            row = dict(frame_index=index, time_s=timestamp, track_id=int(ids[k]),\n                                       baseline_confidence=float(scores[k]), geometry_confidence=float(gs[j]),\n                                       association_iou=float(overlap[i,j]), baseline_box=boxes[k].tolist(),\n                                       geometry_box=gb[j].tolist(), keypoints=kp[j].tolist())\n                            geom_file.write(json.dumps(row, separators=(\',\', \':\'))+\'\\n\'); counts[\'observations\'] += 1\n                            ground_votes[int(ids[k])].append(bool(kp[j,:4,1].mean()>kp[j,4:,1].mean()))\n                        if len(checkpoints) < 16 and timestamp >= resume_s+len(checkpoints)*60:\n                            cv2.imwrite(str(output/f\'geometry_{index:06d}.jpg\'), result.plot())\n                            checkpoints.append(timestamp)\n            if counts[\'detection_frames\'] % 600 == 0:\n                track_file.flush(); geom_file.flush()\n                print(camera, f\'{timestamp:.1f}s / {stop_frame/fps:.1f}s\', dict(counts),\n                      \'runtime\', round(time.monotonic()-start,1), flush=True)\n                # Histories only hold active/recent tracks, rather than an hour of parked IDs.\n                stale = [tid for tid,h in histories.items() if h and timestamp-h[-1][0] > 10]\n                for tid in stale:\n                    del histories[tid]\n            if index % int(fps*300) < fps/config[\'detection_hz\']:\n                cv2.imwrite(str(output/f\'view_{int(timestamp):05d}.jpg\'), frame)\n            index += 1\n    finally:\n        cap.release(); track_file.close(); geom_file.close()\n    ground_fraction=float(np.mean([np.mean(v)>=.5 for v in ground_votes.values()])) if ground_votes else .5\n    ground_indices=list(range(4)) if ground_fraction>=.5 else list(range(4,8))\n    report = dict(camera_id=camera, source_video=video.name, source_sha256=source_sha,\n                  width=width, height=height, fps=fps, frames_reported=total,\n                  frames_decoded=index, expected_decoded_frames=stop_frame,\n                  decode_complete=index==stop_frame, start_s=resume_s, stop_s=stop_frame/fps,\n                  timestamps=\'Monotonic container PTS checked against frame/FPS; fallback when inconsistent\',\n                  timestamp_fallback_frames=timestamp_fallbacks, detection_hz=config[\'detection_hz\'],\n                  geometry_hz=geometry_hz, ground_indices=ground_indices,\n                  ground_order_vote_agreement=max(ground_fraction,1-ground_fraction),\n                  model_sha256=MODEL_SHA, model_revision=MODEL_REVISION, model_file=MODEL_FILE,\n                  counts=dict(counts), runtime_s=time.monotonic()-start,\n                  known_speeds_used=False, benchmark_vehicle_scales_used=False,\n                  id_reset_at_resume_s=resume_s if resume_s else None)\n    (output/\'inference_report.json\').write_text(json.dumps(report, indent=2)+\'\\n\', encoding=\'utf-8\')\n    assert report[\'decode_complete\'], f\'Source decode ended early: {index}/{stop_frame}\'\n    print(\'COMPLETE\', camera, dict(counts), round(report[\'runtime_s\'],1), flush=True)\n\n\nif __name__ == \'__main__\':\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--video\',required=True);p.add_argument(\'--output\',required=True)\n    p.add_argument(\'--camera\',required=True);p.add_argument(\'--duration\',type=float)\n    p.add_argument(\'--resume-s\',type=float,default=0);p.add_argument(\'--device\',default=\'0\')\n    a=p.parse_args();inference(a.video,a.output,a.camera,a.duration,a.resume_s,a.device)\n', 'cctv_camera_profiles.json': '{\n  "schema_version": 1,\n  "description": "Manually selected road regions from each supplied camera view. Metric calibration is inferred from moving cars, without known speeds or road measurements.",\n  "detection_hz": 10,\n  "geometry_hz": 2.5,\n  "calibration_windows_s": [300, 600, 900],\n  "minimum_cars_per_region": 6,\n  "preferred_cars_per_region": 12,\n  "dimension_prior_m": {"length": 4.5, "width": 1.8},\n  "cameras": {\n    "commercial_ii_01": {\n      "display_name": "Commercial II-01",\n      "video": "NVR_ch6_main_20261009170002_20261009180000.mp4",\n      "sha256": "8e4437807fcae7c56f69e6cd55f04374d71e708630afbdfc811b1f192e10beee",\n      "regions": {\n        "left_carriageway": [[0,0.85],[0.30,0.52],[0.43,0.41],[0.44,0.53],[0.43,0.68],[0.56,0.715],[0.57,1],[0,1]],\n        "right_carriageway": [[0.57,0.415],[0.65,0.43],[0.75,0.52],[0.98,0.78],[1,0.92],[1,1],[0.57,1],[0.57,0.715],[0.56,0.60],[0.535,0.45]]\n      },\n      "limitations": "Downhill surface approximated by a separate local plane per carriageway. Parking and the median are excluded by region and motion filters."\n    },\n    "k_block_st20": {\n      "display_name": "K Block St 20 opposite side",\n      "geometry_hz": 10,\n      "motion_history_minimum_span_s": 0.1,\n      "calibration_minimum_observations": 4,\n      "calibration_minimum_span_s": 0.30,\n      "speed_window_s": 1.2,\n      "speed_minimum_span_s": 0.35,\n      "speed_quadratic_minimum_span_s": 0.8,\n      "video": "D02_20261009160351.mp4",\n      "sha256": "bc71067c55f50e21a39da6037617f418282fde1be2bb4d2fe47c6f52525fd5d2",\n      "regions": {\n        "near_approach": [[0,0.55],[0.14,0.29],[0.59,0.25],[0.63,0.32],[0.83,0.34],[0.85,0.84],[0.72,1],[0,1]],\n        "cross_street": [[0,0.40],[0.16,0.25],[0.25,0.19],[0.61,0.11],[0.75,0.10],[0.75,0.18],[0.59,0.25],[0.14,0.29],[0,0.55]]\n      },\n      "limitations": "Junction and near approach calibrated separately. Speeds are local 2D trajectory derivatives, allowing turns and changes in speed."\n    }\n  }\n}\n', 'validation_inference.py': '"""Speed-blind detection and cuboid extraction for constant-speed validation clips."""\nfrom pathlib import Path\nimport argparse\nimport hashlib\nimport json\nimport time\nimport urllib.request\n\nimport cv2\nimport numpy as np\nfrom scipy.optimize import linear_sum_assignment\nfrom ultralytics import YOLO\n\nMODEL_REPO=\'mehmetkeremturkcan/UrbanOmniDetect\'\nMODEL_REVISION=\'2701a2eec9a73afb5ffd852740d125dfcd065ce5\'\nMODEL_FILE=\'checkpoints/urbanomnidetect_yolo11x-p2_1920.pt\'\nMODEL_SHA=\'9aa55280a5ef1c9d89c0acb5baeb974b90fbdf3c1deead1413b24d7af249226d\'\n\n\ndef overlaps(a,b):\n    lo=np.maximum(a[:,None,:2],b[None,:,:2]);hi=np.minimum(a[:,None,2:],b[None,:,2:])\n    intersection=np.prod(np.maximum(0,hi-lo),axis=2)\n    return intersection/(np.prod(a[:,2:]-a[:,:2],axis=1)[:,None]+\n                         np.prod(b[:,2:]-b[:,:2],axis=1)[None,:]-intersection+1e-9)\n\n\ndef infer(folder,output,device=\'0\'):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    model_dir=output.parent/\'models\';model_dir.mkdir(exist_ok=True)\n    weights=model_dir/\'urban_geometry.pt\'\n    if not weights.exists():\n        urllib.request.urlretrieve(f\'https://huggingface.co/{MODEL_REPO}/resolve/{MODEL_REVISION}/{MODEL_FILE}\',weights)\n    assert hashlib.sha256(weights.read_bytes()).hexdigest()==MODEL_SHA\n    geometry_model=YOLO(str(weights))\n    for video in sorted(Path(folder).rglob(\'*.MP4\')):\n        target=output/video.stem;target.mkdir(exist_ok=True)\n        tracker=YOLO(str(model_dir/\'yolo11s.pt\'))\n        cap=cv2.VideoCapture(str(video));assert cap.isOpened()\n        width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH));height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))\n        fps=float(cap.get(cv2.CAP_PROP_FPS));total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n        observations=[];tracks=[];times=[];start=time.monotonic();index=0\n        while True:\n            ok,frame=cap.read()\n            if not ok:break\n            pts=float(cap.get(cv2.CAP_PROP_POS_MSEC))/1000\n            timestamp=pts if abs(pts-index/fps)<.1 else index/fps\n            times.append(timestamp)\n            if index==0:cv2.imwrite(str(target/\'reference.jpg\'),frame)\n            detected=tracker.track(frame,persist=True,tracker=\'bytetrack.yaml\',device=device,\n                                   imgsz=1280,classes=[2,5,7],conf=.15,verbose=False)[0]\n            boxes=detected.boxes.xyxy.cpu().numpy()\n            ids=detected.boxes.id.cpu().numpy().astype(int) if detected.boxes.id is not None else np.full(len(boxes),-1)\n            scores=detected.boxes.conf.cpu().numpy()\n            for box,tid,score in zip(boxes,ids,scores):\n                tracks.append({\'frame_index\':index,\'time_s\':timestamp,\'track_id\':int(tid),\n                               \'box\':box.tolist(),\'confidence\':float(score)})\n            selected=[i for i,box in enumerate(boxes) if box[3]-box[1]>=45 and ids[i]>=0]\n            if selected:\n                result=geometry_model.predict(frame,device=device,imgsz=1920,classes=[0],conf=.15,verbose=False)[0]\n                if result.keypoints is not None and len(result.boxes):\n                    gb=result.boxes.xyxy.cpu().numpy();gs=result.boxes.conf.cpu().numpy()\n                    kp=result.keypoints.data.cpu().numpy()\n                    overlap=overlaps(boxes[selected],gb);ii,jj=linear_sum_assignment(1-overlap)\n                    for i,j in zip(ii,jj):\n                        if overlap[i,j]<.25:continue\n                        k=selected[i]\n                        observations.append({\'frame_index\':index,\'time_s\':timestamp,\'track_id\':int(ids[k]),\n                            \'baseline_confidence\':float(scores[k]),\'geometry_confidence\':float(gs[j]),\n                            \'association_iou\':float(overlap[i,j]),\'baseline_box\':boxes[k].tolist(),\n                            \'geometry_box\':gb[j].tolist(),\'keypoints\':kp[j].tolist()})\n                    if len(observations)%20<2:\n                        cv2.imwrite(str(target/f\'geometry_{index:04d}.jpg\'),result.plot())\n            index+=1\n        cap.release()\n        assert index==total\n        (target/\'observations.json\').write_text(json.dumps(observations),encoding=\'utf-8\')\n        (target/\'tracks.json\').write_text(json.dumps(tracks),encoding=\'utf-8\')\n        if observations:\n            points=np.array([r[\'keypoints\'] for r in observations])\n            vote=points[:,:4,1].mean(axis=1)>points[:,4:,1].mean(axis=1)\n            ground_indices=list(range(4)) if vote.mean()>=.5 else list(range(4,8))\n            agreement=float(max(vote.mean(),1-vote.mean()))\n        else:ground_indices=[];agreement=0.0\n        report={\'source_video\':video.name,\'width\':width,\'height\':height,\'fps\':fps,\'frames\':total,\n                \'duration_s\':total/fps,\'timestamps\':times,\'model_sha256\':MODEL_SHA,\n                \'model_revision\':MODEL_REVISION,\'model_filename\':MODEL_FILE,\'ground_indices\':ground_indices,\n                \'ground_order_vote_agreement\':agreement,\'runtime_s\':time.monotonic()-start,\n                \'observations\':len(observations),\'reference_speeds_used_by_inference\':False}\n        (target/\'geometry_run_report.json\').write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\n        print(video.name,\'frames\',index,\'geometry\',len(observations),\'seconds\',round(report[\'runtime_s\'],1),flush=True)\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--input\',required=True);parser.add_argument(\'--output\',required=True)\n    parser.add_argument(\'--device\',default=\'0\')\n    args=parser.parse_args();infer(args.input,args.output,args.device)\n', 'calibrate_speed.py': '"""Experimental local road rectification from learned cuboids and explicit car-size priors.\n\nIndependent implementation of rectangle/aspect/rigidity fitting, inspired by\nSochor et al. (2017) and the UrbanOmniDetect (2026) ground-footprint approach.\nRanges are sensitivity envelopes, not confidence intervals or accuracy claims.\n"""\nfrom pathlib import Path\nfrom collections import defaultdict\nimport csv\nimport json\nimport math\nimport os\nimport cv2\nimport numpy as np\nfrom scipy.optimize import least_squares\n\nROOT = Path(os.environ.get(\'FYP_SPEED_ROOT\',str(Path(__file__).resolve().parent)))\ndownloaded_geometry = ROOT / \'outputs\' / \'geometry\'\ndefault_geometry = downloaded_geometry if downloaded_geometry.exists() else ROOT / \'output\' / \'inputs\' / \'geometry\'\nGEOMETRY = Path(os.environ.get(\'FYP_GEOMETRY_DIR\', str(default_geometry)))\nOUT = Path(os.environ.get(\'FYP_SPEED_OUTPUT\',str(ROOT / \'outputs\' / \'speed_experiment\')))\nLENGTH_M, WIDTH_M = 4.5, 1.8\nDIMENSION_SCENARIOS = [(4.5,1.8),(3.2,1.45),(3.2,2.0),(5.2,1.45),(5.2,2.0)]\n\n\ndef project(points, H):\n    p = np.asarray(points, dtype=float)\n    flat = p.reshape(-1,2)\n    result = np.c_[flat,np.ones(len(flat))] @ H.T\n    denominator = result[:,2]\n    safe = np.where(np.abs(denominator)<1e-8, np.where(denominator>=0,1e-8,-1e-8),denominator)\n    return (result[:,:2]/safe[:,None]).reshape(p.shape)\n\n\ndef compose(theta, normalizer):\n    p,q,a,b = theta\n    return np.array([[np.exp(a),b,0],[0,1,0],[p,q,1]]) @ normalizer\n\n\ndef measures(quads):\n    edges = np.roll(quads,-1,axis=1)-quads\n    sizes = np.linalg.norm(edges,axis=2)+1e-9\n    unit = edges/sizes[:,:,None]\n    cosines = np.sum(unit*np.roll(unit,-1,axis=1),axis=2)\n    opposite = unit[:,:2,0]*unit[:,2:,1]-unit[:,:2,1]*unit[:,2:,0]\n    lengths=(sizes[:,0]+sizes[:,2])/2\n    widths=(sizes[:,1]+sizes[:,3])/2\n    return cosines,opposite,lengths,widths\n\n\ndef fit_mapping(records, width, height, length_m=LENGTH_M, width_m=WIDTH_M, fast=False, legacy_normalization=False):\n    quads=np.array([r[\'quad\'] for r in records],dtype=float)\n    ids=np.array([r[\'track_id\'] for r in records])\n    unique=np.unique(ids)\n    center=np.median(quads.reshape(-1,2),axis=0)\n    normalizer=np.array([[2/width,0,-2*center[0]/width],[0,2/width,-2*center[1]/width],[0,0,1]])\n    if legacy_normalization:\n        normalizer=np.array([[2/width,0,-1],[0,2/width,-height/width],[0,0,1]])\n    weights=np.array([1/math.sqrt(np.sum(ids==i)) for i in ids])\n    weights/=np.mean(weights)\n    groups=[np.flatnonzero(ids==i) for i in unique]\n\n    def residual(theta):\n        H=compose(theta,normalizer)\n        world=project(quads,H)\n        angles,parallel,lengths,widths=measures(world)\n        rigidity=np.zeros(len(records))\n        for group in groups:\n            rigidity[group]=np.log(lengths[group]/np.median(lengths[group]))\n        shape=np.log((lengths/widths)/(length_m/width_m))\n        geometric=np.c_[angles,parallel*0.5,shape*0.65,rigidity*0.5]*weights[:,None]\n        den=(np.c_[quads.reshape(-1,2),np.ones(len(quads)*4)]@H.T)[:,2]\n        pole=np.maximum(0,0.08-den)*15\n        return np.r_[geometric.ravel(),pole]\n\n    q_starts=([0,1,2] if not fast else [1,2]) if legacy_normalization else ([0,2,5,10,20] if not fast else [2,10])\n    aspect_starts=([0,0.7] if not fast else [0]) if legacy_normalization else ([-3,0] if not fast else [-2])\n    starts=[(0,q,a,0) for q in q_starts for a in aspect_starts]\n    fits=[]\n    for theta in starts:\n        bound=4 if legacy_normalization else 100\n        shape_bound=2.5 if legacy_normalization else 7\n        shear_bound=4 if legacy_normalization else 30\n        result=least_squares(residual,theta,bounds=([-bound,-bound,-shape_bound,-shear_bound],[bound,bound,shape_bound,shear_bound]),\n                             loss=\'soft_l1\',f_scale=0.12,max_nfev=500)\n        fits.append((float(np.mean(residual(result.x)**2)),result))\n    _,result=min(fits,key=lambda item:item[0])\n    H=compose(result.x,normalizer)\n    world=project(quads,H)\n    _,_,lengths,_=measures(world)\n    # Give each track one vote; correlated video frames do not count as independent cars.\n    scale=length_m/np.median([np.median(lengths[g]) for g in groups])\n    H=np.diag([scale,scale,1])@H\n    world=project(quads,H)\n    directions=world[:,1]-world[:,0]\n    directions/=np.linalg.norm(directions,axis=1)[:,None]\n    directions[directions[:,1]<0]*=-1\n    direction=np.median(directions,axis=0)\n    angle=math.pi/2-math.atan2(direction[1],direction[0])\n    rotation=np.array([[math.cos(angle),-math.sin(angle),0],[math.sin(angle),math.cos(angle),0],[0,0,1]])\n    H=rotation@H\n    center=np.median(project(quads,H).mean(axis=1),axis=0)\n    H=np.array([[1,0,-center[0]],[0,1,-center[1]],[0,0,1]])@H\n    H/=np.linalg.norm(H[2,:2]) if np.linalg.norm(H[2,:2])>0 else H[2,2]\n    angle_error,_,lengths,widths=measures(project(quads,H))\n    singular=np.linalg.svd(result.jac,compute_uv=False)\n    diagnostics={\'fit_track_ids\':[int(i) for i in unique],\'fit_observations\':len(records),\n       \'median_angle_error_deg\':float(np.median(np.degrees(np.arcsin(np.clip(np.abs(angle_error),0,1))))),\n       \'median_length_m\':float(np.median(lengths)),\'median_width_m\':float(np.median(widths)),\n       \'parameter_condition_number\':float(singular[0]/max(singular[-1],1e-12)),\n       \'optimizer_success\':bool(result.success),\'theta\':result.x.tolist(),\n       \'dimension_prior_length_m\':length_m,\'dimension_prior_width_m\':width_m}\n    return H,diagnostics\n\n\ndef prepare(observations, indices, width, height, longitudinal_order=\'checkpoint\'):\n    by_id=defaultdict(list)\n    for r in observations:by_id[r[\'track_id\']].append(r)\n    records=[]\n    for tid,group in by_id.items():\n        group.sort(key=lambda r:r[\'time_s\'])\n        centers=np.array([[(r[\'baseline_box\'][0]+r[\'baseline_box\'][2])/2,\n                           (r[\'baseline_box\'][1]+r[\'baseline_box\'][3])/2] for r in group])\n        if len(group)<8 or np.linalg.norm(centers[-1]-centers[0])<80:continue\n        side=\'left\' if np.median(centers[:,0])<width/2 else \'right\'\n        for j,r in enumerate(group):\n            p=np.array(r[\'keypoints\'])[indices]\n            if p.shape[1]>2 and np.min(p[:,2])<0.30:continue\n            if r[\'geometry_confidence\']<0.25 or r[\'association_iou\']<0.40:continue\n            quad=p[:,:2]\n            if not np.isfinite(quad).all():continue\n            if longitudinal_order==\'motion\':\n                quad=quad[np.argsort(np.arctan2(*(quad-quad.mean(axis=0))[:,::-1].T))]\n            if not cv2.isContourConvex(quad.astype(np.float32)):continue\n            if abs(cv2.contourArea(quad.astype(np.float32)))<300:continue\n            if np.min(np.linalg.norm(np.roll(quad,-1,axis=0)-quad,axis=1))<8:continue\n            cx,cy=quad.mean(axis=0)\n            if not (0.20*height<cy<0.82*height and 0.02*width<cx<0.98*width):continue\n            if np.min(quad[:,0])<4 or np.max(quad[:,0])>width-4 or np.max(quad[:,1])>height-4:continue\n            if longitudinal_order==\'motion\':\n                lo=max(0,j-3);hi=min(len(group)-1,j+3)\n                motion=centers[hi]-centers[lo]\n                if np.linalg.norm(motion)<5:continue\n                edge=np.roll(quad,-1,axis=0)-quad\n                alignment=np.abs(edge@motion)/(np.linalg.norm(edge,axis=1)*np.linalg.norm(motion)+1e-9)\n                scores=[np.mean(alignment[[0,2]]),np.mean(alignment[[1,3]])]\n                if abs(scores[0]-scores[1])<0.08:continue\n                if scores[1]>scores[0]:quad=np.roll(quad,-1,axis=0)\n            elif longitudinal_order!=\'checkpoint\':\n                raise ValueError(\'Unknown longitudinal corner-order method.\')\n            records.append({\'track_id\':int(tid),\'time_s\':r[\'time_s\'],\'frame_index\':r[\'frame_index\'],\n                            \'side\':side,\'quad\':quad.tolist()})\n    return records\n\n\ndef estimate_track(records,H):\n    times=np.array([r[\'time_s\'] for r in records],dtype=float)\n    quads=np.array([r[\'quad\'] for r in records],dtype=float)\n    world=project(quads,H)\n    centers=world.mean(axis=1)\n    t=times-times.mean()\n    initial=np.r_[np.polyfit(t,centers[:,0],1),np.polyfit(t,centers[:,1],1)]\n    def residual(p):return np.r_[(p[0]*t+p[1]-centers[:,0]),(p[2]*t+p[3]-centers[:,1])]\n    fitted=least_squares(residual,initial,loss=\'soft_l1\',f_scale=0.4)\n    speed=float(np.hypot(fitted.x[0],fitted.x[2])*3.6)\n    rms=float(np.sqrt(np.mean(residual(fitted.x)**2)))\n    angles,_,lengths,widths=measures(world)\n    length_cv=float(1.4826*np.median(np.abs(lengths-np.median(lengths)))/np.median(lengths))\n    angle_error=float(np.median(np.degrees(np.arcsin(np.clip(np.abs(angles),0,1)))))\n    midpoint=(times[0]+times[-1])/2\n    half_speeds=[]\n    for mask in [times<=midpoint,times>midpoint]:\n        if mask.sum()>=4:\n            velocity=np.array([np.polyfit(times[mask]-times[mask].mean(),centers[mask,k],1)[0] for k in [0,1]])\n            half_speeds.append(float(np.linalg.norm(velocity)*3.6))\n    drift=abs(half_speeds[0]-half_speeds[1])/max(speed,1) if len(half_speeds)==2 else None\n    return {\'experimental_kmh\':speed,\'trajectory_fit_rms_m\':rms,\'length_relative_mad\':length_cv,\n            \'angle_error_deg\':angle_error,\'half_span_speed_relative_difference\':drift,\n            \'median_inferred_length_m\':float(np.median(lengths)),\'median_inferred_width_m\':float(np.median(widths))}\n\n\ndef sample_balanced(records, maximum=10):\n    groups=defaultdict(list)\n    for r in records:groups[r[\'track_id\']].append(r)\n    return [group[i] for group in groups.values() for i in np.unique(np.linspace(0,len(group)-1,min(len(group),maximum)).astype(int))]\n\n\ndef fit_road_mappings(records, report):\n    width,height=report[\'width\'],report[\'height\']\n    calibration={}\n    for side in [\'left\',\'right\']:\n        local=[r for r in records if r[\'side\']==side]\n        ids=sorted(set(r[\'track_id\'] for r in local))\n        if len(ids)<3:\n            calibration[side]={\'accepted\':False,\'reason\':\'Fewer than three qualifying car tracks\',\'track_ids\':ids}\n            continue\n        # Deterministic holdout by track identity; held-out predicted boxes are consistency evidence only.\n        held_ids=ids[::3]\n        fit_ids=[i for i in ids if i not in held_ids]\n        if len(fit_ids)<3:fit_ids=ids;held_ids=[]\n        fitting=sample_balanced([r for r in local if r[\'track_id\'] in fit_ids])\n        H,diagnostics=fit_mapping(fitting,width,height)\n        alternatives=[]\n        for length_m,width_m in DIMENSION_SCENARIOS[1:]:\n            alternative,_=fit_mapping(fitting,width,height,length_m,width_m,fast=True)\n            alternatives.append(alternative)\n        rng=np.random.default_rng(42)\n        for _ in range(10):\n            selected=set(rng.choice(fit_ids,size=max(3,round(len(fit_ids)*0.75)),replace=False).tolist())\n            subset=[r for r in fitting if r[\'track_id\'] in selected]\n            alternative,_=fit_mapping(subset,width,height,fast=True)\n            alternatives.append(alternative)\n        held=[r for r in local if r[\'track_id\'] in held_ids]\n        held_angle=None\n        if held:\n            angles,_,_,_=measures(project(np.array([r[\'quad\'] for r in held]),H))\n            held_angle=float(np.median(np.degrees(np.arcsin(np.clip(np.abs(angles),0,1)))))\n        accepted=diagnostics[\'median_angle_error_deg\']<20 and (held_angle is None or held_angle<25)\n        accepted=accepted and report[\'ground_order_vote_agreement\']>=0.8 and diagnostics[\'parameter_condition_number\']<1e5\n        diagnostics.update({\'accepted\':bool(accepted),\'held_out_track_ids\':held_ids,\n          \'held_out_median_angle_error_deg\':held_angle,\'H_image_to_experimental_meters\':H.tolist(),\n          \'alternative_mappings\':[a.tolist() for a in alternatives],\n          \'interpretation\':\'Approximate local plane and dimension-prior mapping; no measured accuracy.\'})\n        calibration[side]=diagnostics\n        diagnostics[\'support_hull_image\'] = cv2.convexHull(\n            np.array([r[\'quad\'] for r in local], dtype=np.float32).reshape(-1, 2)).reshape(-1, 2).tolist()\n    return calibration\n\n\ndef calculate_speed_rows(records, calibration):\n    speed_rows=[]\n    for side in [\'left\',\'right\']:\n        diagnostics=calibration.get(side,{})\n        if not diagnostics.get(\'accepted\'):\n            continue\n        H=np.array(diagnostics[\'H_image_to_experimental_meters\'])\n        alternatives=[np.array(a) for a in diagnostics[\'alternative_mappings\']]\n        hull=np.array(diagnostics[\'support_hull_image\'], dtype=np.float32)\n        # Restrict reuse to the road region supported by the calibration footprints.\n        local=[r for r in records if r[\'side\']==side and all(\n            cv2.pointPolygonTest(hull, (float(p[0]),float(p[1])), True)>=-0.01 for p in r[\'quad\'])]\n        ids=sorted(set(r[\'track_id\'] for r in local))\n        for tid in ids:\n            group=sorted([r for r in local if r[\'track_id\']==tid],key=lambda r:r[\'time_s\'])\n            if len(group)<8 or group[-1][\'time_s\']-group[0][\'time_s\']<1.2:continue\n            nominal=estimate_track(group,H)\n            values=[nominal[\'experimental_kmh\']]+[estimate_track(group,a)[\'experimental_kmh\'] for a in alternatives]\n            low,high=float(min(values)),float(max(values))\n            sensitivity=(high-low)/max(nominal[\'experimental_kmh\'],1)\n            reasons=[]\n            if nominal[\'length_relative_mad\']>0.25:reasons.append(\'predicted_size_inconsistent\')\n            if nominal[\'trajectory_fit_rms_m\']>0.8:reasons.append(\'trajectory_not_consistent_with_local_plane\')\n            if nominal[\'angle_error_deg\']>25:reasons.append(\'footprint_not_rectangular_after_mapping\')\n            if nominal[\'half_span_speed_relative_difference\'] is not None and nominal[\'half_span_speed_relative_difference\']>0.35:reasons.append(\'speed_changes_or_geometry_drift_across_span\')\n            if sensitivity>0.8:reasons.append(\'high_calibration_sensitivity\')\n            speed_rows.append({\'track_id\':tid,\'side\':side,\'time_start_s\':group[0][\'time_s\'],\n              \'time_end_s\':group[-1][\'time_s\'],\'geometry_observations\':len(group),\n              \'quality_passed\':not bool(reasons),\'quality_reasons\':\';\'.join(reasons),\n              \'dimension_and_fit_sensitivity_low_kmh\':low,\'dimension_and_fit_sensitivity_high_kmh\':high,\n              **nominal})\n    return speed_rows\n\n\ndef run(calibration=None, geometry_dir=None, output_dir=None, calibration_reused=None):\n    geometry_dir=Path(geometry_dir) if geometry_dir is not None else GEOMETRY\n    output_dir=Path(output_dir) if output_dir is not None else OUT\n    output_dir.mkdir(parents=True,exist_ok=True)\n    report=json.loads((geometry_dir/\'geometry_run_report.json\').read_text())\n    observations=json.loads((geometry_dir/\'observations.json\').read_text())\n    records=prepare(observations,report[\'ground_indices\'],report[\'width\'],report[\'height\'])\n    reused=(calibration is not None) if calibration_reused is None else calibration_reused\n    if calibration is None:\n        calibration=fit_road_mappings(records, report)\n    before_support_filter=len(records)\n    records=[r for r in records if calibration.get(r[\'side\'],{}).get(\'accepted\') and all(\n        cv2.pointPolygonTest(np.array(calibration[r[\'side\']][\'support_hull_image\'],dtype=np.float32),\n                             (float(p[0]),float(p[1])),True)>=-0.01 for p in r[\'quad\'])]\n    speed_rows=calculate_speed_rows(records, calibration)\n    (output_dir/\'filtered_ground_footprints.json\').write_text(json.dumps(records),encoding=\'utf-8\')\n    (output_dir/\'calibration.json\').write_text(json.dumps(calibration,indent=2),encoding=\'utf-8\')\n    fields=[\'track_id\',\'side\',\'time_start_s\',\'time_end_s\',\'geometry_observations\',\n            \'quality_passed\',\'quality_reasons\',\'dimension_and_fit_sensitivity_low_kmh\',\n            \'dimension_and_fit_sensitivity_high_kmh\',\'experimental_kmh\',\'trajectory_fit_rms_m\',\n            \'length_relative_mad\',\'angle_error_deg\',\'half_span_speed_relative_difference\',\n            \'median_inferred_length_m\',\'median_inferred_width_m\']\n    for name, rows in [(\'experimental_speeds.csv\',speed_rows),\n                       (\'qualified_experimental_speeds.csv\',[r for r in speed_rows if r[\'quality_passed\']])]:\n        with (output_dir/name).open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n            writer=csv.DictWriter(f,fieldnames=fields)\n            writer.writeheader()\n            writer.writerows(rows)\n    api_results=[{\'track_id\':r[\'track_id\'],\'carriageway\':r[\'side\'],\n      \'time_interval_s\':[r[\'time_start_s\'],r[\'time_end_s\']],\n      \'experimental_kmh\':r[\'experimental_kmh\'],\n      \'estimated_kmh\':r[\'experimental_kmh\'],\n      \'sensitivity_range_kmh\':[r[\'dimension_and_fit_sensitivity_low_kmh\'],r[\'dimension_and_fit_sensitivity_high_kmh\']],\n      \'range_type\':\'dimension_and_calibration_sensitivity\',\'accuracy_validated\':False,\n      \'method\':\'learned_3d_ground_footprints_with_assumed_vehicle_dimensions\'}\n      for r in speed_rows if r[\'quality_passed\']]\n    (output_dir/\'vehicle_speed_estimates.json\').write_text(json.dumps(api_results,indent=2),encoding=\'utf-8\')\n    result={\'calibration_reused\':reused,\'source_geometry_observations\':len(observations),\'filtered_ground_footprints\':len(records),\n      \'footprints_outside_usable_calibration\':before_support_filter-len(records),\n      \'geometry_model_filename\':report[\'model_filename\'],\'geometry_model_sha256\':report[\'model_sha256\'],\n      \'ground_corner_indices\':report[\'ground_indices\'],\n      \'experimental_track_estimates\':len(speed_rows),\'quality_passed_track_estimates\':sum(r[\'quality_passed\'] for r in speed_rows),\n      \'nominal_car_length_m\':LENGTH_M,\'nominal_car_width_m\':WIDTH_M,\'dimension_scenarios_m\':DIMENSION_SCENARIOS,\n      \'range_type\':\'Dimension-prior and track-subset sensitivity envelope. Not a statistical confidence interval.\',\n      \'scope\':\'Local visible road regions on each carriageway; passenger-car hypotheses only.\',\n      \'accuracy_validated\':False,\'speed_units\':\'km/h under assumed vehicle dimensions\',\n      \'limitations\':[\'No independent speed or road-distance ground truth.\',\n        \'Ground corners are model predictions; hidden corners can be wrong.\',\n        \'A local planar road and approximately constant speed over each fitted span are assumed.\',\n        \'Dimension priors are explicit analyst assumptions, not recognized make/model dimensions.\',\n        \'Sensitivity range does not bound unmodelled errors such as lens distortion or road grade.\']}\n    (output_dir/\'speed_run_report.json\').write_text(json.dumps(result,indent=2),encoding=\'utf-8\')\n    print(json.dumps(result,indent=2))\n    for r in speed_rows:print(r[\'track_id\'],r[\'side\'],round(r[\'experimental_kmh\'],1),r[\'quality_passed\'],r[\'quality_reasons\'])\n    return calibration,speed_rows,records\n\n\nif __name__==\'__main__\':run()\n', 'cctv_calibration.py': '"""Camera-specific multi-car calibration with explicit geometric stability checks.\n\nSelects an affine or projective road map by held-out car geometry, then tests\nsubsets of independent tracks. These checks measure consistency, not speed\naccuracy. A generic passenger-car dimension prior supplies the metric scale.\n"""\nimport argparse\nfrom collections import Counter, defaultdict\nimport gzip\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nfrom scipy.optimize import least_squares\n\nfrom calibrate_speed import fit_mapping, measures, project, sample_balanced,compose\n\n\ndef read_rows(folder, name):\n    with gzip.open(Path(folder)/name, \'rt\', encoding=\'utf-8\') as stream:\n        return [json.loads(line) for line in stream if line.strip()]\n\n\ndef region_for(point, profile, width, height):\n    for name, polygon in profile[\'regions\'].items():\n        contour = (np.asarray(polygon)*[width,height]).astype(np.float32)\n        if cv2.pointPolygonTest(contour, tuple(map(float, point)), False) >= 0:\n            return name\n    return None\n\n\ndef motion_qualified(group, minimum_observations=6, minimum_span_s=1.5):\n    if len(group) < minimum_observations:\n        return False\n    times=np.array([r[\'time_s\'] for r in group])\n    centers=np.array([np.asarray(r[\'quad\']).mean(axis=0) for r in group])\n    extent=float(np.max(np.linalg.norm(centers-np.median(centers,axis=0),axis=1))*2)\n    return bool(times.max()-times.min() >= minimum_span_s-.001 and extent >= 60)\n\n\ndef ground_order(observations):\n    """The pinned checkpoint predates the current model card\'s corner convention.\n\n    Its output is eight XY points, with ground corners 0:4 in these recordings.\n    Select the lower projected face using one aggregate vote per moving track.\n    Preserve edge topology within the selected face.\n    """\n    votes=defaultdict(list)\n    for r in observations:\n        p=np.asarray(r[\'keypoints\'],float)\n        if p.ndim==2 and p.shape[0]==8 and p.shape[1]>=2:\n            votes[r[\'track_id\']].append(bool(p[:4,1].mean()>p[4:,1].mean()))\n    if not votes:return dict(indices=[],agreement=0.0,tracks=0)\n    fraction=float(np.mean([np.mean(v)>=.5 for v in votes.values()]))\n    return dict(indices=list(range(4)) if fraction>=.5 else list(range(4,8)),\n                agreement=max(fraction,1-fraction),tracks=len(votes),\n                method=\'Lower projected face, one aggregate vote per moving track; checkpoint edge topology retained\')\n\n\ndef footprints(observations, profile, width, height, moving_only=True, ground_indices=None):\n    if ground_indices is None:ground_indices=ground_order(observations)[\'indices\']\n    if len(ground_indices)!=4:return [],dict(no_ground_face=len(observations))\n    groups=defaultdict(list); counts=Counter()\n    for r in observations:\n        p=np.asarray(r[\'keypoints\'],float)[ground_indices]\n        if p.shape[0]!=4 or p.shape[1] not in [2,3] or not np.isfinite(p).all():\n            counts[\'nonfinite_or_wrong_shape\']+=1;continue\n        if (p.shape[1]==3 and p[:,2].min()<.30) or r[\'geometry_confidence\']<.25 or r[\'association_iou\']<.40:\n            counts[\'low_confidence\']+=1;continue\n        quad=p[:,:2]; edges=np.linalg.norm(np.roll(quad,-1,axis=0)-quad,axis=1)\n        if not cv2.isContourConvex(quad.astype(np.float32)) or abs(cv2.contourArea(quad.astype(np.float32)))<250 or edges.min()<7:\n            counts[\'degenerate_quad\']+=1;continue\n        if quad[:,0].min()<4 or quad[:,0].max()>width-4 or quad[:,1].min()<4 or quad[:,1].max()>height-4:\n            counts[\'clipped\']+=1;continue\n        region=region_for(quad.mean(axis=0),profile,width,height)\n        if not region:\n            counts[\'outside_road\']+=1;continue\n        row=dict(track_id=int(r[\'track_id\']),time_s=r[\'time_s\'],frame_index=r[\'frame_index\'],\n                 region=region,quad=quad.tolist(),confidence=float(p[:,2].min()) if p.shape[1]==3 else None)\n        groups[(region,int(r[\'track_id\']))].append(row)\n    accepted=[]\n    for group in groups.values():\n        group.sort(key=lambda r:r[\'time_s\'])\n        if moving_only and not motion_qualified(group,profile.get(\'calibration_minimum_observations\',6),\n                                                 profile.get(\'calibration_minimum_span_s\',1.5)):\n            counts[\'stationary_or_short_track_observations\']+=len(group);continue\n        accepted.extend(group)\n    counts[\'accepted_observations\']=len(accepted)\n    counts[\'accepted_tracks\']=len(set(r[\'track_id\'] for r in accepted))\n    return accepted,dict(counts)\n\n\ndef fit_affine(records,width,height,length_m=4.5,width_m=1.8):\n    quads=np.array([r[\'quad\'] for r in records]);ids=np.array([r[\'track_id\'] for r in records])\n    groups=[np.flatnonzero(ids==i) for i in np.unique(ids)]\n    center=np.median(quads.reshape(-1,2),axis=0)\n    normalizer=np.array([[2/width,0,-2*center[0]/width],[0,2/width,-2*center[1]/width],[0,0,1]])\n    weights=np.array([1/np.sqrt(np.sum(ids==i)) for i in ids]);weights/=weights.mean()\n    def compose(theta):\n        return np.array([[np.exp(theta[0]),theta[1],0],[0,1,0],[0,0,1]])@normalizer\n    def residual(theta):\n        angles,parallel,lengths,widths=measures(project(quads,compose(theta)))\n        return (np.c_[angles,parallel*.5,np.log((lengths/widths)/(length_m/width_m))*.65]*weights[:,None]).ravel()\n    result=least_squares(residual,[-1,0],bounds=([-7,-30],[7,30]),loss=\'soft_l1\',f_scale=.12,max_nfev=500)\n    H=compose(result.x)\n    lengths=measures(project(quads,H))[2]\n    scale=length_m/np.median([np.median(lengths[g]) for g in groups])\n    H=np.diag([scale,scale,1])@H\n    singular=np.linalg.svd(result.jac,compute_uv=False)\n    return H,dict(optimizer_success=bool(result.success),parameter_condition_number=float(singular[0]/max(singular[-1],1e-12)))\n\n\ndef fit_metric_projective(records,width,height):\n    """Robust fleet-size prior also constrains perspective scale with road depth.\n\n    The original angle/rigidity fit fixes only the overall median length, which\n    can leave large position-dependent size errors. This version adds soft\n    length and width priors across independent cars. Unknown true car dimensions\n    remain a source of systematic error.\n    """\n    quads=np.array([r[\'quad\'] for r in records]);ids=np.array([r[\'track_id\'] for r in records])\n    unique=np.unique(ids);groups=[np.flatnonzero(ids==i) for i in unique]\n    center=np.median(quads.reshape(-1,2),axis=0)\n    normalizer=np.array([[2/width,0,-2*center[0]/width],[0,2/width,-2*center[1]/width],[0,0,1]])\n    _,initial=fit_mapping(records,width,height,fast=True)\n    theta=np.array(initial[\'theta\']);H=compose(theta,normalizer)\n    lengths=measures(project(quads,H))[2]\n    scale=4.5/np.median([np.median(lengths[g]) for g in groups])\n    weights=np.array([1/np.sqrt(np.sum(ids==i)) for i in ids]);weights/=weights.mean()\n    def mapping(v):return np.diag([np.exp(v[4]),np.exp(v[4]),1])@compose(v[:4],normalizer)\n    def residual(v):\n        H=mapping(v);world=project(quads,H)\n        angles,parallel,lengths,widths=measures(world)\n        rigidity=np.zeros(len(records))\n        for group in groups:rigidity[group]=np.log(lengths[group]/np.median(lengths[group]))\n        geometric=np.c_[angles,parallel*.5,np.log((lengths/widths)/2.5)*.65,rigidity*.5,\n                         np.log(lengths/4.5)*.40,np.log(widths/1.8)*.40]*weights[:,None]\n        den=(np.c_[quads.reshape(-1,2),np.ones(len(quads)*4)]@H.T)[:,2]\n        return np.r_[geometric.ravel(),np.maximum(0,.08-den)*15]\n    result=least_squares(residual,np.r_[theta,np.log(scale)],bounds=([-100,-100,-7,-30,-12],[100,100,7,30,12]),\n                         loss=\'soft_l1\',f_scale=.12,max_nfev=500)\n    singular=np.linalg.svd(result.jac,compute_uv=False)\n    return mapping(result.x),dict(optimizer_success=bool(result.success),\n                                 parameter_condition_number=float(singular[0]/max(singular[-1],1e-12)),\n                                 soft_dimension_prior_m=dict(length=4.5,width=1.8),prior_weight=.40)\n\n\ndef fit_method(method,records,width,height,fast=False):\n    if method==\'affine\':return fit_affine(records,width,height)\n    if method==\'projective_metric\':return fit_metric_projective(records,width,height)\n    return fit_mapping(records,width,height,fast=fast)\n\n\ndef geometry_quality(records,H):\n    if not records:\n        return dict(angle_error_deg=None,shape_log_error=None,length_relative_mad=None,score=1e6)\n    groups=defaultdict(list)\n    for r in records:groups[r[\'track_id\']].append(r)\n    angles=[];shapes=[];rigidity=[];metric_error=[]\n    for group in groups.values():\n        a,_,lengths,widths=measures(project(np.array([r[\'quad\'] for r in group]),H))\n        angles.append(float(np.median(np.degrees(np.arcsin(np.clip(abs(a),0,1))))))\n        shapes.append(float(np.median(abs(np.log((lengths/widths)/2.5)))))\n        rigidity.append(float(1.4826*np.median(abs(lengths-np.median(lengths)))/np.median(lengths)))\n        metric_error.append(float(np.median(abs(np.log(lengths/4.5))+abs(np.log(widths/1.8)))/2))\n    angle,shape,rigid=map(lambda x:float(np.median(x)),[angles,shapes,rigidity])\n    metric=float(np.median(metric_error))\n    return dict(angle_error_deg=angle,shape_log_error=shape,length_relative_mad=rigid,\n                typical_car_dimension_log_error=metric,score=angle+10*shape+20*rigid+10*metric)\n\n\ndef scale_change(H,alternative,records):\n    """Compare local metric magnification; arbitrary map rotations do not matter."""\n    centers=np.array([np.mean(r[\'quad\'],axis=0) for r in records])\n    ratios=[]\n    for delta in [[3,0],[0,3]]:\n        a=np.linalg.norm(project(centers+delta,H)-project(centers,H),axis=1)\n        b=np.linalg.norm(project(centers+delta,alternative)-project(centers,alternative),axis=1)\n        ratios.extend((abs(b/np.maximum(a,1e-9)-1)).tolist())\n    return float(np.median(ratios)),float(np.quantile(ratios,.90))\n\n\ndef fit_region(records,width,height,minimum=6):\n    ids=sorted(set(r[\'track_id\'] for r in records))\n    if len(ids)<minimum:\n        return dict(accepted=False,reason=f\'Only {len(ids)} independent moving car tracks; need {minimum}\',track_ids=ids)\n    held=ids[::4];fit_ids=[i for i in ids if i not in held]\n    fitting=sample_balanced([r for r in records if r[\'track_id\'] in fit_ids],maximum=10)\n    testing=sample_balanced([r for r in records if r[\'track_id\'] in held],maximum=10)\n    candidates=[]\n    for method in [\'affine\',\'projective\',\'projective_metric\']:\n        H,diag=fit_method(method,fitting,width,height)\n        fit_quality=geometry_quality(fitting,H);held_quality=geometry_quality(testing,H)\n        candidates.append(dict(method=method,H=H,diagnostics=diag,fit_quality=fit_quality,held_quality=held_quality,\n                               score=held_quality[\'score\']+.1*fit_quality[\'score\']))\n    selected=min(candidates,key=lambda r:r[\'score\'])\n    method=selected[\'method\']\n    # Refit selected geometry on all qualifying calibration cars after model selection.\n    fitting=sample_balanced(records,maximum=10)\n    H,diag=fit_method(method,fitting,width,height)\n    alternatives=[];rng=np.random.default_rng(20261009)\n    subset_changes=[]\n    for _ in range(6):\n        chosen=set(rng.choice(ids,size=max(4,int(np.ceil(len(ids)*.75))),replace=False).tolist())\n        subset=[r for r in fitting if r[\'track_id\'] in chosen]\n        alt,_=fit_method(method,subset,width,height,fast=True)\n        alternatives.append(alt);subset_changes.append(scale_change(H,alt,fitting)[0])\n    hull=cv2.convexHull(np.array([r[\'quad\'] for r in records],np.float32).reshape(-1,2)).reshape(-1,2)\n    den=np.c_[hull,np.ones(len(hull))]@H[2]\n    finite_plane=bool(np.isfinite(H).all() and not (den.min()<=0<=den.max()))\n    quality=geometry_quality(fitting,H)\n    held_quality=selected[\'held_quality\']\n    bootstrap=float(np.quantile(subset_changes,.90))\n    reasons=[]\n    if not finite_plane:reasons.append(\'Projective pole intersects calibration support\')\n    if quality[\'angle_error_deg\']>20 or held_quality[\'angle_error_deg\']>25:reasons.append(\'Rectangle consistency is poor\')\n    if held_quality[\'length_relative_mad\']>.30:reasons.append(\'Held-out footprint scale changes along tracks\')\n    if bootstrap>.35:reasons.append(\'Metric mapping changes by more than 35% between car subsets\')\n    if diag[\'parameter_condition_number\']>1e5:reasons.append(\'Ill-conditioned geometric fit\')\n    # Scale uncertainty is explicit. This is a sensitivity envelope, not a confidence interval.\n    alternatives.extend([np.diag([factor,factor,1])@H for factor in [.80,1.20]])\n    candidate_report=[{k:v for k,v in r.items() if k!=\'H\'} for r in candidates]\n    return dict(accepted=not reasons,reason=\'; \'.join(reasons) if reasons else \'Passed internal geometry and subset-stability checks\',\n                method=method,track_ids=ids,observations=len(records),held_out_track_ids=held,\n                candidate_comparison=candidate_report,quality=quality,selected_held_quality=held_quality,\n                subset_scale_changes=subset_changes,subset_scale_change_p90=bootstrap,\n                H_image_to_experimental_meters=H.tolist(),alternative_mappings=[a.tolist() for a in alternatives],\n                support_hull_image=hull.tolist(),diagnostics=diag,\n                dimension_prior_m=dict(length=4.5,width=1.8),\n                interpretation=\'Experimental metric scale from unknown passenger-car dimensions; accuracy unvalidated\')\n\n\ndef calibrate(inputs,camera,output):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    config_path=Path(__file__).parent/\'cctv_camera_profiles.json\'\n    config=json.loads(config_path.read_text());profile=config[\'cameras\'][camera]\n    observations=[];reports=[]\n    for folder in inputs:\n        report=json.loads((Path(folder)/\'inference_report.json\').read_text());reports.append(report)\n        assert report[\'source_sha256\']==profile[\'sha256\'] and report[\'camera_id\']==camera\n        observations.extend(read_rows(folder,\'observations.jsonl.gz\'))\n    width,height=reports[0][\'width\'],reports[0][\'height\']\n    order=ground_order([r for r in observations if r[\'time_s\']<max(config[\'calibration_windows_s\'])])\n    assert order[\'agreement\']>=.8,\'Inconsistent projected ground-face ordering\'\n    regions={};windows=[];previous={}\n    for end_s in config[\'calibration_windows_s\']:\n        if end_s>max(r[\'stop_s\'] for r in reports):continue\n        records,filters=footprints([r for r in observations if r[\'time_s\']<end_s],profile,width,height,ground_indices=order[\'indices\'])\n        window=dict(end_s=end_s,filters=filters,regions={})\n        for name in profile[\'regions\']:\n            if name in regions:continue\n            local=[r for r in records if r[\'region\']==name]\n            result=fit_region(local,width,height,config[\'minimum_cars_per_region\'])\n            result[\'calibration_end_s\']=end_s\n            if result.get(\'accepted\'):\n                stability=None\n                if name in previous and previous[name].get(\'accepted\'):\n                    stability=scale_change(np.array(previous[name][\'H_image_to_experimental_meters\']),\n                                           np.array(result[\'H_image_to_experimental_meters\']),sample_balanced(local))[0]\n                result[\'scale_change_from_previous_window\']=stability\n                preferred=len(result[\'track_ids\'])>=config[\'preferred_cars_per_region\'] and result[\'subset_scale_change_p90\']<=.20\n                stable=stability is not None and stability<=.15 and result[\'subset_scale_change_p90\']<=.25\n                if preferred or stable or end_s==config[\'calibration_windows_s\'][-1]:\n                    result[\'selection_reason\']=\'Enough independent cars and stable subsets\' if preferred else \'Stable across calibration windows\' if stable else \'Longest configured window; passed minimum consistency checks\'\n                    regions[name]=result\n            previous[name]=result\n            window[\'regions\'][name]=result\n            print(camera,name,end_s,\'cars\',len(result[\'track_ids\']),\'accepted\',result.get(\'accepted\'),result.get(\'reason\'),flush=True)\n        windows.append(window)\n        if len(regions)==len(profile[\'regions\']):break\n    for name in profile[\'regions\']:\n        if name not in regions:\n            regions[name]=previous.get(name,dict(accepted=False,reason=\'No calibration observations\',track_ids=[]))\n    chosen_end=max([r.get(\'calibration_end_s\',0) for r in regions.values()]+[300])\n    records,filters=footprints([r for r in observations if r[\'time_s\']<chosen_end],profile,width,height,ground_indices=order[\'indices\'])\n    with gzip.open(output/\'calibration_footprints.jsonl.gz\',\'wt\',encoding=\'utf-8\') as stream:\n        for r in records:stream.write(json.dumps(r,separators=(\',\',\':\'))+\'\\n\')\n    report=dict(schema_version=2,camera_id=camera,source_video=profile[\'video\'],source_sha256=profile[\'sha256\'],\n                width=width,height=height,fps=reports[0][\'fps\'],profile=profile,\n                profile_sha256=hashlib.sha256(config_path.read_bytes()).hexdigest(),\n                regions=regions,window_comparison=windows,calibration_duration_s=chosen_end,\n                ground_order=order,keypoint_confidence_available=bool(observations and len(observations[0][\'keypoints\'][0])==3),\n                all_regions_accepted=all(r.get(\'accepted\') for r in regions.values()),\n                known_speeds_used=False,accuracy_validated=False,benchmark_vehicle_scales_used=False,\n                cache_policy=\'Camera-specific mapping; profile, source hash and pipeline version must match. Reuse requires a consistent static view.\',\n                dimension_sensitivity=\'Uniform scale 0.8 to 1.2, plus independent-car subset variation; not a statistical confidence interval\',\n                source_sha256_by_file={name:hashlib.sha256((Path(__file__).parent/name).read_bytes()).hexdigest()\n                                      for name in [\'cctv_calibration.py\',\'calibrate_speed.py\']})\n    (output/\'calibration.json\').write_text(json.dumps(report,indent=2,allow_nan=False)+\'\\n\',encoding=\'utf-8\')\n    print(\'Calibration complete:\',camera,\'seconds\',chosen_end,\'all regions accepted\',report[\'all_regions_accepted\'],flush=True)\n    return report\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser(description=__doc__);p.add_argument(\'--inputs\',nargs=\'+\')\n    p.add_argument(\'--input\');p.add_argument(\'--camera\',required=True);p.add_argument(\'--output\',required=True)\n    a=p.parse_args();calibrate(a.inputs or [a.input],a.camera,a.output)\n', 'cctv_speed.py': '"""Local road-plane speeds for CCTV, including turning and accelerating cars.\n\nSpeeds are robust local polynomial derivatives of projected ground-footprint\ncenters. Each track is split at long gaps and region changes. No constant-speed\nassumption or expected-speed correction is applied.\n"""\nimport argparse\nfrom collections import Counter,defaultdict\nimport csv\nimport gzip\nimport json\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\n\nfrom calibrate_speed import project,measures\nfrom cctv_calibration import footprints,read_rows\nfrom camera_cache import compare_views\n\n\ndef local_velocity(times,centers,at,window_s=2.4,minimum_span_s=1.0,quadratic_minimum_span_s=1.5):\n    times=np.asarray(times,float);centers=np.asarray(centers,float)\n    mask=abs(times-at)<=window_s/2+1e-6\n    t=times[mask]-at;p=centers[mask]\n    if len(t)<4 or np.ptp(t)<minimum_span_s-.001:return None\n    degree=2 if len(t)>=5 and np.ptp(t)>=quadratic_minimum_span_s-.001 and (t<-.001).sum()>=2 and (t>.001).sum()>=2 else 1\n    design=np.stack([t**j for j in range(degree+1)],axis=1)\n    initial=np.exp(-.5*(t/(window_s/2))**2)\n    weights=initial.copy()\n    coefficients=None\n    for _ in range(4):\n        root=np.sqrt(weights)\n        coefficients=np.linalg.lstsq(design*root[:,None],p*root[:,None],rcond=None)[0]\n        error=np.linalg.norm(design@coefficients-p,axis=1)\n        weights=initial*np.minimum(1,.25/np.maximum(error,1e-9))\n    residual=float(np.sqrt(np.average(np.sum((design@coefficients-p)**2,axis=1),weights=weights)))\n    return dict(speed_kmh=float(np.linalg.norm(coefficients[1])*3.6),\n                velocity_m_s=coefficients[1].tolist(),rms_m=residual,n=len(t),span_s=float(np.ptp(t)),\n                polynomial_degree=degree)\n\n\ndef split_segments(records,max_gap_s=1.25):\n    result=[];segment=[]\n    for r in sorted(records,key=lambda r:r[\'time_s\']):\n        if segment and (r[\'time_s\']-segment[-1][\'time_s\']>max_gap_s or r[\'region\']!=segment[-1][\'region\']):\n            result.append(segment);segment=[]\n        segment.append(r)\n    if segment:result.append(segment)\n    return result\n\n\ndef view_checks(inputs,calibration):\n    reference=cv2.imread(str(Path(inputs[0])/\'reference.jpg\'),0)\n    assert reference is not None\n    reference=cv2.resize(reference,(960,round(reference.shape[0]*960/reference.shape[1])))\n    rows=[]\n    for folder in inputs:\n        for path in sorted(Path(folder).glob(\'view_*.jpg\')):\n            frame=cv2.imread(str(path),0)\n            frame=cv2.resize(frame,(reference.shape[1],reference.shape[0]))\n            comparison=compare_views(reference,frame)\n            rows.append(dict(time_s=int(path.stem.rsplit(\'_\',1)[1]),comparison=comparison))\n    rows.sort(key=lambda r:r[\'time_s\'])\n    return rows\n\n\ndef calculate(inputs,calibration_file,output):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    calibration=json.loads(Path(calibration_file).read_text())\n    observations=[];inference=[]\n    for folder in inputs:\n        report=json.loads((Path(folder)/\'inference_report.json\').read_text());inference.append(report)\n        assert report[\'camera_id\']==calibration[\'camera_id\'] and report[\'source_sha256\']==calibration[\'source_sha256\']\n        observations.extend(read_rows(folder,\'observations.jsonl.gz\'))\n    records,filters=footprints(observations,calibration[\'profile\'],calibration[\'width\'],calibration[\'height\'],\n                              moving_only=False,ground_indices=calibration[\'ground_order\'][\'indices\'])\n    profile=calibration[\'profile\']\n    velocity_options=dict(window_s=profile.get(\'speed_window_s\',2.4),minimum_span_s=profile.get(\'speed_minimum_span_s\',1.0),\n                          quadratic_minimum_span_s=profile.get(\'speed_quadratic_minimum_span_s\',1.5))\n    checks=view_checks(inputs,calibration)\n    invalid_intervals=[(max(0,r[\'time_s\']-300),r[\'time_s\']+300) for r in checks if not r[\'comparison\'][\'valid\']]\n    groups=defaultdict(list);counts=Counter();rows=[]\n    for r in records:groups[r[\'track_id\']].append(r)\n    for tid,group in groups.items():\n        for segment in split_segments(group):\n            name=segment[0][\'region\'];mapping=calibration[\'regions\'][name]\n            if not mapping.get(\'accepted\'):\n                counts[\'unaccepted_region\']+=len(segment);continue\n            H=np.array(mapping[\'H_image_to_experimental_meters\']);hull=np.array(mapping[\'support_hull_image\'],np.float32)\n            # Filtering can create gaps. Split again instead of fitting through missing support.\n            supported=[]\n            for r in segment:\n                if any(lo<=r[\'time_s\']<=hi for lo,hi in invalid_intervals):\n                    counts[\'view_change\']+=1;continue\n                if not all(cv2.pointPolygonTest(hull,tuple(map(float,p)),True)>=-2 for p in r[\'quad\']):\n                    counts[\'outside_calibration_support\']+=1;continue\n                supported.append(r)\n            for sub in split_segments(supported):\n                if len(sub)<4:\n                    counts[\'short_segment\']+=len(sub);continue\n                times=np.array([r[\'time_s\'] for r in sub]);quads=np.array([r[\'quad\'] for r in sub])\n                world=project(quads,H);centers=world.mean(axis=1)\n                angles,_,lengths,widths=measures(world)\n                angle_error=np.median(np.degrees(np.arcsin(np.clip(abs(angles),0,1))),axis=1)\n                alternative_centers=[project(quads,np.array(a)).mean(axis=1) for a in mapping[\'alternative_mappings\']]\n                for i,r in enumerate(sub):\n                    nominal=local_velocity(times,centers,times[i],**velocity_options)\n                    if nominal is None:\n                        counts[\'insufficient_local_window\']+=1;continue\n                    mask=abs(times-times[i])<=velocity_options[\'window_s\']/2+1e-6\n                    size_mad=float(1.4826*np.median(abs(lengths[mask]-np.median(lengths[mask])))/np.median(lengths[mask]))\n                    reasons=[]\n                    if nominal[\'rms_m\']>.8:reasons.append(\'trajectory_residual\')\n                    if np.median(angle_error[mask])>25:reasons.append(\'quad_angle_error\')\n                    if size_mad>.25:reasons.append(\'unstable_footprint_length\')\n                    if not (2.5<=np.median(lengths[mask])<=6.5 and 1.1<=np.median(widths[mask])<=2.6):\n                        reasons.append(\'implausible_car_footprint_size\')\n                    alternative=[local_velocity(times,p,times[i],**velocity_options) for p in alternative_centers]\n                    values=[nominal[\'speed_kmh\']]+[v[\'speed_kmh\'] for v in alternative if v is not None]\n                    low,high=min(values),max(values)\n                    if (high-low)/max(nominal[\'speed_kmh\'],2)>.85:reasons.append(\'excessive_mapping_sensitivity\')\n                    if reasons:\n                        counts.update(reasons);continue\n                    row=dict(camera_id=calibration[\'camera_id\'],track_id=int(tid),region=name,\n                             frame_index=r[\'frame_index\'],time_s=r[\'time_s\'],experimental_kmh=nominal[\'speed_kmh\'],\n                             sensitivity_low_kmh=low,sensitivity_high_kmh=high,\n                             trajectory_rms_m=nominal[\'rms_m\'],window_observations=nominal[\'n\'],\n                             window_span_s=nominal[\'span_s\'],polynomial_degree=nominal[\'polynomial_degree\'],\n                             length_relative_mad=size_mad,ground_center_image=np.mean(r[\'quad\'],axis=0).tolist(),\n                             ground_center_world_m=centers[i].tolist(),quad=r[\'quad\'])\n                    rows.append(row);counts[\'accepted_speed_samples\']+=1\n    rows.sort(key=lambda r:(r[\'frame_index\'],r[\'track_id\']))\n    with gzip.open(output/\'speeds.jsonl.gz\',\'wt\',encoding=\'utf-8\') as stream:\n        for r in rows:stream.write(json.dumps(r,separators=(\',\',\':\'))+\'\\n\')\n    keys=[\'camera_id\',\'track_id\',\'region\',\'frame_index\',\'time_s\',\'experimental_kmh\',\'sensitivity_low_kmh\',\n          \'sensitivity_high_kmh\',\'trajectory_rms_m\',\'window_observations\',\'window_span_s\',\'polynomial_degree\',\'length_relative_mad\']\n    with (output/\'speed_samples.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as stream:\n        writer=csv.DictWriter(stream,fieldnames=keys);writer.writeheader()\n        writer.writerows({k:r[k] for k in keys} for r in rows)\n    track_summaries=[]\n    by_id=defaultdict(list)\n    for r in rows:by_id[r[\'track_id\']].append(r)\n    for tid,local in sorted(by_id.items()):\n        speeds=np.array([r[\'experimental_kmh\'] for r in local])\n        track_summaries.append(dict(track_id=tid,start_s=local[0][\'time_s\'],end_s=local[-1][\'time_s\'],\n                                    samples=len(local),median_experimental_kmh=float(np.median(speeds)),\n                                    min_experimental_kmh=float(speeds.min()),max_experimental_kmh=float(speeds.max())))\n    (output/\'track_summary.json\').write_text(json.dumps(track_summaries,indent=2)+\'\\n\')\n    report=dict(camera_id=calibration[\'camera_id\'],source_sha256=calibration[\'source_sha256\'],\n                total_duration_s=max(r[\'stop_s\'] for r in inference),inference_reports=inference,\n                footprint_filters=filters,speed_counts=dict(counts),cars_with_speed_estimates=len(by_id),\n                static_view_checks=checks,invalid_view_intervals_s=invalid_intervals,\n                all_sampled_views_consistent=all(r[\'comparison\'][\'valid\'] for r in checks),\n                method=\'Robust local polynomial derivative of 2D road-plane ground centroid\',\n                local_velocity_options=velocity_options,\n                latency_note=f\'Offline centered window uses future observations up to {velocity_options["window_s"]/2} seconds\',\n                display_note=\'Decimal experimental km/h; dimension and car-subset sensitivity range is secondary\',\n                motorcycle_bus_truck_policy=\'Tracked but no metric label: this checkpoint supplies car ground footprints only\',\n                car_size_screen_m=dict(length=[2.5,6.5],width=[1.1,2.6]),\n                accuracy_validated=False,known_speeds_used=False,\n                count_note=\'Track IDs can fragment or restart at 900s; counts are tracks, not guaranteed unique vehicles\')\n    (output/\'speed_report.json\').write_text(json.dumps(report,indent=2,allow_nan=False)+\'\\n\',encoding=\'utf-8\')\n    print(\'Speed complete:\',calibration[\'camera_id\'],\'car tracks\',len(by_id),\'samples\',len(rows),dict(counts),flush=True)\n    return report\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser(description=__doc__);p.add_argument(\'--inputs\',nargs=\'+\',required=True)\n    p.add_argument(\'--calibration\',required=True);p.add_argument(\'--output\',required=True)\n    a=p.parse_args();calculate(a.inputs,a.calibration,a.output)\n', 'cctv_render.py': '"""Render calibration replay and complete CCTV speed videos without AVI buffers.\n\nFull source frame rates are retained. Speed recordings are split into ten-minute\nMP4s so the complete output can be reviewed on GitHub within its file-size limit.\n"""\nimport argparse\nfrom bisect import bisect_left\nfrom collections import defaultdict\nimport json\nfrom pathlib import Path\nimport subprocess\n\nimport cv2\nimport numpy as np\n\nfrom calibrate_speed import project\nfrom cctv_calibration import read_rows\n\n\nCOLORS=[(40,220,240),(255,190,70),(160,220,100)]\n\n\ndef clock(seconds):\n    seconds=max(0,int(seconds));return f\'{seconds//3600:02d}:{seconds//60%60:02d}:{seconds%60:02d}\'\n\n\ndef grid_overlay(calibration,size):\n    width,height=size;scale=np.array([width/calibration[\'width\'],height/calibration[\'height\']])\n    grid=np.zeros((height,width,3),np.uint8);regions=[]\n    for i,(name,mapping) in enumerate(calibration[\'regions\'].items()):\n        color=COLORS[i%len(COLORS)]\n        if not mapping.get(\'accepted\'):continue\n        H=np.asarray(mapping[\'H_image_to_experimental_meters\']);hull=np.asarray(mapping[\'support_hull_image\'])\n        polygon=np.round(hull*scale).astype(np.int32)\n        mask=np.zeros((height,width),np.uint8);cv2.fillPoly(mask,[polygon],255)\n        road=np.zeros_like(mask)\n        roi=np.round(np.asarray(calibration[\'profile\'][\'regions\'][name])*[width,height]).astype(np.int32)\n        cv2.fillPoly(road,[roi],255);mask=cv2.bitwise_and(mask,road)\n        world=project(hull,H);lo=np.floor(world.min(axis=0)/5)*5;hi=np.ceil(world.max(axis=0)/5)*5\n        layer=np.zeros_like(grid)\n        for axis in [0,1]:\n            values=np.arange(lo[axis],hi[axis]+.1,5)\n            if len(values)>120:values=np.linspace(lo[axis],hi[axis],60)\n            for value in values:\n                line=np.array([[lo[0],lo[1]],[hi[0],hi[1]]])\n                line[:,axis]=value\n                points=project(line,np.linalg.inv(H))*scale\n                if np.isfinite(points).all() and abs(points).max()<1e7:\n                    cv2.line(layer,tuple(np.round(points[0]).astype(int)),tuple(np.round(points[1]).astype(int)),color,1,cv2.LINE_AA)\n        grid[mask>0]=layer[mask>0]\n        border=np.zeros_like(grid);cv2.polylines(border,[polygon],True,color,2,cv2.LINE_AA)\n        grid[(mask>0)&np.any(border>0,axis=2)]=border[(mask>0)&np.any(border>0,axis=2)]\n        regions.append(dict(region=name,support_world_span_m=(world.max(axis=0)-world.min(axis=0)).tolist(),\n                            calibration_s=mapping[\'calibration_end_s\'],car_tracks=len(mapping[\'track_ids\'])))\n    return grid,regions\n\n\ndef label(frame,text,point,color,small=False):\n    font=.45 if small else .62;thickness=1 if small else 2\n    width,height=frame.shape[1],frame.shape[0]\n    (tw,th),baseline=cv2.getTextSize(text,cv2.FONT_HERSHEY_SIMPLEX,font,thickness)\n    x=max(2,min(int(point[0]),width-tw-6));y=max(th+3,min(int(point[1]),height-baseline-4))\n    cv2.rectangle(frame,(x-2,y-th-3),(x+tw+3,y+baseline+2),(15,15,15),-1)\n    cv2.putText(frame,text,(x,y),cv2.FONT_HERSHEY_SIMPLEX,font,color,thickness,cv2.LINE_AA)\n\n\ndef sample_at(records,times,time_s,max_distance=.30):\n    j=bisect_left(times,time_s)\n    candidates=[k for k in [j-1,j] if 0<=k<len(records)]\n    if not candidates:return None\n    k=min(candidates,key=lambda k:abs(times[k]-time_s))\n    if abs(times[k]-time_s)>max_distance:return None\n    return records[k]\n\n\ndef encoder(path,width,height,fps,segmented=False):\n    args=[\'ffmpeg\',\'-hide_banner\',\'-loglevel\',\'error\',\'-y\',\'-f\',\'rawvideo\',\'-pix_fmt\',\'bgr24\',\n          \'-s\',f\'{width}x{height}\',\'-r\',str(fps),\'-i\',\'pipe:0\',\'-an\',\'-c:v\',\'libx264\',\'-preset\',\'fast\',\n          \'-crf\',\'30\',\'-threads\',\'2\',\'-pix_fmt\',\'yuv420p\']\n    if segmented:\n        args+=[\'-force_key_frames\',\'expr:gte(t,n_forced*600)\',\'-f\',\'segment\',\'-segment_time\',\'600\',\n               \'-reset_timestamps\',\'1\',\'-segment_format_options\',\'movflags=+faststart\']\n    else:args+=[\'-movflags\',\'+faststart\']\n    return subprocess.Popen(args+[str(path)],stdin=subprocess.PIPE,stderr=subprocess.PIPE)\n\n\ndef render(video,inputs,calibration_file,speed_folder,output,duration=None):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    calibration=json.loads(Path(calibration_file).read_text());speed_folder=Path(speed_folder)\n    speed_report=json.loads((speed_folder/\'speed_report.json\').read_text())\n    tracks=defaultdict(list);observations=defaultdict(list);speeds=defaultdict(list)\n    detection_frames=set()\n    for folder in inputs:\n        for r in read_rows(folder,\'tracks.jsonl.gz\'):\n            tracks[r[\'frame_index\']].append(r);detection_frames.add(r[\'frame_index\'])\n        for r in read_rows(folder,\'observations.jsonl.gz\'):observations[r[\'track_id\']].append(r)\n    for r in read_rows(speed_folder,\'speeds.jsonl.gz\'):speeds[r[\'track_id\']].append(r)\n    for group in list(speeds.values())+list(observations.values()):group.sort(key=lambda r:r[\'time_s\'])\n    speed_times={tid:[r[\'time_s\'] for r in group] for tid,group in speeds.items()}\n    observation_times={tid:[r[\'time_s\'] for r in group] for tid,group in observations.items()}\n    cap=cv2.VideoCapture(str(video));assert cap.isOpened()\n    fps=float(cap.get(cv2.CAP_PROP_FPS));total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n    expected_frames=total if duration is None else min(total,int(round(duration*fps)))\n    width,height=1280,720;scale=np.array([width/calibration[\'width\'],height/calibration[\'height\']])\n    grid,region_info=grid_overlay(calibration,(width,height))\n    grid_mask=np.any(grid>0,axis=2)\n    calibration_frames=min(expected_frames,int(round(calibration[\'calibration_duration_s\']*fps)))\n    speed_encoder=encoder(output/\'speed_%02d.mp4\',width,height,fps,True)\n    calibration_encoder=encoder(output/\'calibration.mp4\',width,height,fps)\n    current=[];index=0;labelled_frames=0;labelled_ids=set();examples=[];counts={}\n    name=calibration[\'profile\'][\'display_name\']\n    region_colors={k:COLORS[i%len(COLORS)] for i,k in enumerate(calibration[\'regions\'])}\n    try:\n        while index<expected_frames:\n            ok,source=cap.read()\n            if not ok:break\n            time_s=index/fps\n            base=cv2.resize(source,(width,height),interpolation=cv2.INTER_AREA)\n            if index in tracks:current=tracks[index]\n            shown=[r for r in current if index-r[\'frame_index\']<=fps*.16 and r.get(\'region\')]\n            speed_frame=base.copy();n=0\n            for r in shown:\n                tid=r[\'track_id\'];box=np.round(np.array(r[\'box\'])*np.r_[scale,scale]).astype(int)\n                speed=sample_at(speeds[tid],speed_times.get(tid,[]),time_s,.35) if tid in speeds else None\n                color=(70,220,110) if speed else (170,170,170)\n                cv2.rectangle(speed_frame,tuple(box[:2]),tuple(box[2:]),color,2 if speed else 1)\n                if speed:\n                    value=f\'#{tid}  {speed["experimental_kmh"]:.1f} km/h\'\n                    label(speed_frame,value,(box[0],box[1]-24),color)\n                    label(speed_frame,f\'approx {speed["sensitivity_low_kmh"]:.1f}-{speed["sensitivity_high_kmh"]:.1f}\',(box[0],box[1]-5),color,True)\n                    n+=1;labelled_ids.add(tid)\n                elif r.get(\'moving\'):\n                    label(speed_frame,f\'#{tid} tracked | no qualified speed\',(box[0],box[1]-5),color,True)\n            if n:labelled_frames+=1\n            cv2.rectangle(speed_frame,(0,0),(width,90),(12,12,12),-1)\n            cv2.putText(speed_frame,f\'SPEED ESTIMATES | {name} | {clock(time_s)} / {clock(total/fps)}\',(14,29),cv2.FONT_HERSHEY_SIMPLEX,.65,(255,255,255),2,cv2.LINE_AA)\n            cv2.putText(speed_frame,\'Experimental km/h | car-dimension prior | accuracy not independently measured\',(14,58),cv2.FONT_HERSHEY_SIMPLEX,.49,(100,215,255),1,cv2.LINE_AA)\n            speed_encoder.stdin.write(speed_frame.tobytes())\n            if n and (not examples or time_s-examples[-1][\'time_s\']>=300):\n                path=output/f\'speed_example_{len(examples):02d}.jpg\';cv2.imwrite(str(path),speed_frame)\n                examples.append(dict(time_s=time_s,file=path.name,labels=n))\n            if index<calibration_frames:\n                calibration_frame=base.copy()\n                calibration_frame[grid_mask]=cv2.addWeighted(base,.35,grid,.65,0)[grid_mask]\n                for r in shown:\n                    tid=r[\'track_id\'];group=observations.get(tid,[])\n                    obs=sample_at(group,observation_times.get(tid,[]),time_s,.25) if group else None\n                    if obs is None:continue\n                    quad=np.array(obs[\'keypoints\'])[calibration[\'ground_order\'][\'indices\'],:2]*scale\n                    color=region_colors.get(r[\'region\'],(180,180,180))\n                    cv2.polylines(calibration_frame,[np.round(quad).astype(np.int32)],True,color,2,cv2.LINE_AA)\n                    for j,point in enumerate(quad):\n                        point=tuple(np.round(point).astype(int));cv2.circle(calibration_frame,point,4,color,-1)\n                        cv2.putText(calibration_frame,str(j+1),(point[0]+4,point[1]-4),cv2.FONT_HERSHEY_SIMPLEX,.4,color,1,cv2.LINE_AA)\n                cv2.rectangle(calibration_frame,(0,0),(width,100),(12,12,12),-1)\n                cv2.putText(calibration_frame,f\'CALIBRATION REPLAY | {name} | {clock(time_s)} / {clock(calibration_frames/fps)}\',(14,28),cv2.FONT_HERSHEY_SIMPLEX,.61,(255,255,255),2,cv2.LINE_AA)\n                text=\' | \'.join(f\'{k.replace("_"," ")}: {len(v["track_ids"])} cars, {v.get("method","unavailable")}\' for k,v in calibration[\'regions\'].items())\n                cv2.putText(calibration_frame,text,(14,55),cv2.FONT_HERSHEY_SIMPLEX,.46,(100,215,255),1,cv2.LINE_AA)\n                cv2.putText(calibration_frame,\'Final fitted road grid: 5 m spacing | car ground nodes 1-4 | prior 4.5 x 1.8 m\',(14,82),cv2.FONT_HERSHEY_SIMPLEX,.46,(225,225,225),1,cv2.LINE_AA)\n                calibration_encoder.stdin.write(calibration_frame.tobytes())\n                if index==min(calibration_frames-1,int(60*fps)):\n                    cv2.imwrite(str(output/\'calibration_example.jpg\'),calibration_frame)\n            if index==calibration_frames-1:\n                calibration_encoder.stdin.close()\n                error=calibration_encoder.stderr.read().decode(errors=\'replace\');code=calibration_encoder.wait()\n                assert code==0,error\n            index+=1\n            if index%int(fps*300)==0:\n                print(\'Rendered\',calibration[\'camera_id\'],clock(time_s),\'of\',clock(total/fps),\'labelled frames\',labelled_frames,flush=True)\n    finally:\n        cap.release()\n        if not speed_encoder.stdin.closed:speed_encoder.stdin.close()\n        if not calibration_encoder.stdin.closed:calibration_encoder.stdin.close()\n    error=speed_encoder.stderr.read().decode(errors=\'replace\');code=speed_encoder.wait();assert code==0,error\n    if calibration_encoder.poll() is None:\n        error=calibration_encoder.stderr.read().decode(errors=\'replace\');code=calibration_encoder.wait();assert code==0,error\n    assert index==expected_frames,f\'Source decoded {index}/{expected_frames} frames\'\n    files=[]\n    for path in sorted(output.glob(\'*.mp4\')):\n        result=subprocess.run([\'ffprobe\',\'-v\',\'error\',\'-select_streams\',\'v:0\',\'-show_entries\',\n                               \'stream=nb_frames,r_frame_rate,duration,width,height\',\'-of\',\'json\',str(path)],capture_output=True,text=True,check=True)\n        stream=json.loads(result.stdout)[\'streams\'][0]\n        files.append(dict(file=path.name,bytes=path.stat().st_size,**stream))\n    speed_files=[r for r in files if r[\'file\'].startswith(\'speed_\')]\n    rendered_frames=sum(int(r[\'nb_frames\']) for r in speed_files)\n    assert rendered_frames==expected_frames,(rendered_frames,expected_frames)\n    report=dict(camera_id=calibration[\'camera_id\'],source_sha256=calibration[\'source_sha256\'],source_frames=total,\n                speed_rendered_frames=rendered_frames,calibration_rendered_frames=calibration_frames,\n                source_fps=fps,output_resolution=[width,height],audio_included=False,files=files,\n                video_covers_entire_source=expected_frames==total,segment_duration_s=600,\n                render_source_sha256=__import__(\'hashlib\').sha256(Path(__file__).read_bytes()).hexdigest(),\n                frames_with_metric_labels=labelled_frames,labelled_car_tracks=len(labelled_ids),\n                examples=examples,calibration_regions=region_info,\n                calibration_video_note=\'Offline replay shows the final mapping fitted from the selected initial interval.\',\n                speeds_note=\'Only qualified car footprints receive a metric label; grey tracks lack a qualified speed.\',\n                accuracy_validated=False)\n    (output/\'video_manifest.json\').write_text(json.dumps(report,indent=2)+\'\\n\',encoding=\'utf-8\')\n    print(\'All videos complete:\',calibration[\'camera_id\'],rendered_frames,\'speed frames\',calibration_frames,\'calibration frames\',flush=True)\n    return report\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser(description=__doc__);p.add_argument(\'--video\',required=True)\n    p.add_argument(\'--inputs\',nargs=\'+\',required=True);p.add_argument(\'--calibration\',required=True)\n    p.add_argument(\'--speed\',required=True);p.add_argument(\'--output\',required=True)\n    p.add_argument(\'--duration\',type=float,help=\'Optional QA preview duration; omit for full coverage\')\n    a=p.parse_args();render(a.video,a.inputs,a.calibration,a.speed,a.output,a.duration)\n', 'camera_cache.py': '"""Conservative camera-view checks and a versioned, integrity-checked calibration cache."""\nimport hashlib\nimport json\nimport re\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\n\nSCHEMA_VERSION = 1\n\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef canonical_hash(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()\n\n\ndef read_views(video, start_s, duration_s):\n    capture = cv2.VideoCapture(str(video))\n    try:\n        if not capture.isOpened():\n            raise ValueError(\'Cannot decode source video.\')\n        width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))\n        height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))\n        fps = float(capture.get(cv2.CAP_PROP_FPS))\n        frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))\n        if fps <= 0 or min(width, height, frames) <= 0:\n            raise ValueError(\'Invalid video metadata.\')\n        start, end = round(start_s * fps), round((start_s + duration_s) * fps)\n        if start < 0 or end <= start or end > frames:\n            raise ValueError(\'Requested interval is outside the source video.\')\n        views = []\n        indices = np.unique(np.linspace(start, end - 1, 5).astype(int))\n        for index in indices:\n            capture.set(cv2.CAP_PROP_POS_FRAMES, int(index))\n            ok, frame = capture.read()\n            if not ok:\n                raise ValueError(f\'Cannot decode view-check frame {index}.\')\n            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)\n            views.append(cv2.resize(gray, (960, round(height * 960 / width))))\n        return {\'width\': width, \'height\': height, \'fps\': fps, \'frames\': frames,\n                \'start_frame\': start, \'end_frame\': end,\n                \'view_check_frames\': indices.tolist()}, views\n    finally:\n        capture.release()\n\n\ndef compare_views(reference, current):\n    """Reject uncertain matches as well as sufficiently large shifts, zooms or rotations.\n\n    This is a sampled visual consistency check, not proof that the camera never moved.\n    Exclude timestamps, borders and foreground overlays; RANSAC discounts moving vehicles.\n    """\n    if reference is None or current is None or reference.shape != current.shape:\n        return {\'valid\': False, \'reason\': \'view_image_shape_changed\'}\n    height, width = reference.shape\n    mask = np.zeros_like(reference, dtype=np.uint8)\n    mask[round(.15 * height):round(.88 * height), round(.03 * width):round(.97 * width)] = 255\n    detector = cv2.ORB_create(nfeatures=5000, fastThreshold=12)\n    kp1, desc1 = detector.detectAndCompute(reference, mask)\n    kp2, desc2 = detector.detectAndCompute(current, mask)\n    if desc1 is None or desc2 is None:\n        return {\'valid\': False, \'reason\': \'insufficient_static_features\'}\n    pairs = cv2.BFMatcher(cv2.NORM_HAMMING).knnMatch(desc1, desc2, k=2)\n    matches = [a for pair in pairs if len(pair) == 2 for a, b in [pair] if a.distance < .7 * b.distance]\n    if len(matches) < 25:\n        return {\'valid\': False, \'reason\': \'insufficient_view_matches\', \'matches\': len(matches)}\n    a = np.float32([kp1[m.queryIdx].pt for m in matches])\n    b = np.float32([kp2[m.trainIdx].pt for m in matches])\n    matrix, inliers = cv2.findHomography(a, b, cv2.RANSAC, 3.0)\n    if matrix is None or inliers is None or not np.isfinite(matrix).all():\n        return {\'valid\': False, \'reason\': \'view_alignment_failed\'}\n    selected = inliers.ravel().astype(bool)\n    count = int(selected.sum())\n    coverage = min(abs(cv2.contourArea(cv2.convexHull(p[selected]))) / (width * height)\n                   for p in [a, b]) if count >= 3 else 0\n    corners = np.float32([[0, 0], [width, 0], [width, height], [0, height]])\n    projected = cv2.perspectiveTransform(corners[None], matrix)[0]\n    displacement = float(np.max(np.linalg.norm((projected - corners) / [width, height], axis=1)))\n    valid = count >= 25 and count / len(matches) >= .5 and coverage >= .10 and displacement <= .012\n    return {\'valid\': bool(valid), \'reason\': \'consistent_view\' if valid else \'view_changed_or_uncertain\',\n            \'matches\': len(matches), \'inliers\': count, \'feature_coverage_fraction\': coverage,\n            \'max_normalized_corner_displacement\': displacement}\n\n\ndef stable_view(views):\n    comparisons = [compare_views(views[0], view) for view in views[1:]]\n    return {\'valid\': bool(comparisons) and all(item[\'valid\'] for item in comparisons),\n            \'comparisons\': comparisons,\n            \'limitation\': \'Five sampled frames cannot exclude brief camera movement between samples.\'}\n\n\ndef valid_calibration(calibration):\n    if not isinstance(calibration, dict):\n        return False\n    accepted = 0\n    try:\n        for side in [\'left\', \'right\']:\n            mapping = calibration.get(side, {})\n            if not mapping.get(\'accepted\'):\n                continue\n            accepted += 1\n            matrices = [mapping[\'H_image_to_experimental_meters\'], *mapping[\'alternative_mappings\']]\n            if len(matrices) < 2:\n                return False\n            for value in matrices:\n                matrix = np.array(value, dtype=float)\n                if matrix.shape != (3, 3) or not np.isfinite(matrix).all() or abs(np.linalg.det(matrix)) < 1e-12:\n                    return False\n            hull = np.array(mapping[\'support_hull_image\'], dtype=np.float32)\n            if hull.ndim != 2 or hull.shape[1] != 2 or len(hull) < 3 or not np.isfinite(hull).all():\n                return False\n            if not cv2.isContourConvex(hull) or abs(cv2.contourArea(hull)) < 300:\n                return False\n    except (KeyError, TypeError, ValueError, cv2.error):\n        return False\n    return accepted > 0\n\n\nclass CalibrationCache:\n    def __init__(self, directory, camera_id):\n        if not camera_id.strip():\n            raise ValueError(\'A nonempty camera ID is required.\')\n        slug = re.sub(r\'[^a-zA-Z0-9_-]+\', \'-\', camera_id).strip(\'-\')[:48] or \'camera\'\n        self.camera_id = camera_id\n        self.directory = Path(directory) / (slug + \'-\' + canonical_hash(camera_id)[:12])\n        self.path = self.directory / \'calibration_cache.json\'\n\n    def load(self, signature, views):\n        if not self.path.exists():\n            return None, {\'status\': \'miss\', \'reason\': \'no_saved_calibration\'}\n        try:\n            entry = json.loads(self.path.read_text(encoding=\'utf-8\'))\n            if entry[\'schema_version\'] != SCHEMA_VERSION or entry[\'camera_id\'] != self.camera_id:\n                return None, {\'status\': \'miss\', \'reason\': \'cache_schema_or_camera_changed\'}\n            if entry[\'signature\'] != signature:\n                return None, {\'status\': \'miss\', \'reason\': \'resolution_model_or_calibration_settings_changed\'}\n            if canonical_hash(entry[\'calibration\']) != entry[\'calibration_sha256\'] or not valid_calibration(entry[\'calibration\']):\n                return None, {\'status\': \'miss\', \'reason\': \'invalid_calibration_data\'}\n            view_path = self.directory / \'reference_view.png\'\n            if file_sha256(view_path) != entry[\'reference_sha256\']:\n                return None, {\'status\': \'miss\', \'reason\': \'invalid_reference_image\'}\n            reference = cv2.imread(str(view_path), cv2.IMREAD_GRAYSCALE)\n            checks = [compare_views(reference, current) for current in views]\n            if not all(check[\'valid\'] for check in checks):\n                return None, {\'status\': \'miss\', \'reason\': \'camera_view_changed_or_uncertain\', \'view_checks\': checks}\n            return entry[\'calibration\'], {\'status\': \'hit\', \'reason\': \'saved_calibration_matches_camera_view\',\n                                          \'view_checks\': checks}\n        except (OSError, ValueError, KeyError, TypeError, cv2.error):\n            return None, {\'status\': \'miss\', \'reason\': \'cache_unreadable_or_corrupt\'}\n\n    def save(self, signature, reference, calibration, provenance):\n        if not valid_calibration(calibration):\n            raise ValueError(\'No usable road mapping; calibration will not be cached.\')\n        self.directory.mkdir(parents=True, exist_ok=True)\n        view_path = self.directory / \'reference_view.png\'\n        if not cv2.imwrite(str(view_path), reference):\n            raise OSError(\'Could not save camera reference.\')\n        entry = {\'schema_version\': SCHEMA_VERSION, \'camera_id\': self.camera_id,\n                 \'signature\': signature, \'calibration\': calibration,\n                 \'calibration_sha256\': canonical_hash(calibration),\n                 \'reference_sha256\': file_sha256(view_path), \'provenance\': provenance,\n                 \'accuracy_validated\': False}\n        temporary = self.path.with_suffix(\'.tmp\')\n        temporary.write_text(json.dumps(entry, indent=2, allow_nan=False), encoding=\'utf-8\')\n        temporary.replace(self.path)\n'}
camera='commercial_ii_01'
video_name='NVR_ch6_main_20261009170002_20261009180000.mp4'
root=Path('/kaggle/working/pipeline');root.mkdir(exist_ok=True)
for name,source in sources.items():(root/name).write_text(source,encoding='utf-8')
subprocess.check_call([sys.executable,'-m','pip','install','-q','ultralytics==8.4.175','lap==0.5.13'])
videos=list(Path('/kaggle/input').rglob(video_name));assert len(videos)==1,(video_name,videos)
out=Path('/kaggle/working/cctv_outputs')/camera;out.mkdir(parents=True,exist_ok=True)
def run(script,*args):subprocess.check_call([sys.executable,str(root/script),*map(str,args)])

pilot_matches=list(Path('/kaggle/input').rglob(camera+'/pilot/inference_report.json'))
assert len(pilot_matches)==1,pilot_matches
pilot=pilot_matches[0].parent
run('cctv_calibration.py','--input',pilot,'--camera',camera,'--output',out/'calibration')
run('cctv_inference.py','--video',videos[0],'--output',out/'remainder','--camera',camera,'--resume-s',900)
run('cctv_speed.py','--inputs',pilot,out/'remainder','--calibration',out/'calibration/calibration.json','--output',out/'speed')
run('cctv_render.py','--video',videos[0],'--inputs',pilot,out/'remainder','--calibration',out/'calibration/calibration.json','--speed',out/'speed','--output',out/'videos')

import torch,cv2,ultralytics
report={'camera_id':camera,'torch':torch.__version__,'opencv':cv2.__version__,
        'ultralytics':ultralytics.__version__,'gpu':torch.cuda.get_device_name(),
        'source_sha256':{name:hashlib.sha256((root/name).read_bytes()).hexdigest() for name in sources}}
(out/'runtime.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print('Job complete:',camera)
